# Read saved outputs: Hourly equipment health with delayed and corrected readings

Uses the same contract and destination. No generation or table writes. Run the generator with persistence first.

In [ ]:
import json
import re
from pyspark.sql import functions as F
spark.conf.set('spark.sql.session.timeZone', 'UTC')
cfg = json.loads('{\n  "schema_version": "1.0",\n  "title": "Hourly equipment health with delayed and corrected readings",\n  "seed": 42,\n  "task_kind": "analysis",\n  "decisions": [\n    {\n      "key": "objective",\n      "value": "Correct hourly equipment-health averages and late-reading inclusion.",\n      "status": "confirmed",\n      "source": "User prompt",\n      "impact": "Business outcome.",\n      "severity": "requirements",\n      "bindings": {}\n    },\n    {\n      "key": "deliverable",\n      "value": "Databricks-ready PySpark pipeline notebook with plans, checks, and distributed execution explanations.",\n      "status": "confirmed",\n      "source": "User prompt",\n      "impact": "Artifact scope.",\n      "severity": "requirements",\n      "bindings": {\n        "outputs": [\n          "readings_raw",\n          "resolved_readings",\n          "hourly_health",\n          "hourly_comparison",\n          "late_quarantine"\n        ]\n      }\n    },\n    {\n      "key": "grain",\n      "value": "One logical reading per reading_id; physical repeats and corrected versions share identity.",\n      "status": "confirmed",\n      "source": "User prompt and answer",\n      "impact": "Identity resolution.",\n      "severity": "requirements",\n      "bindings": {}\n    },\n    {\n      "key": "metric",\n      "value": "Per machine and UTC event hour averages and independent valid/missing counts for temperature and vibration.",\n      "status": "confirmed",\n      "source": "User answer",\n      "impact": "Aggregation.",\n      "severity": "requirements",\n      "bindings": {\n        "operations": [\n          {\n            "kind": "filter",\n            "name": "eligible_readings",\n            "input": "readings_raw",\n            "predicate": "arrival_delay_minutes <= 120"\n          },\n          {\n            "kind": "deduplicate",\n            "name": "resolved_readings",\n            "input": "eligible_readings",\n            "keys": [\n              "reading_id"\n            ],\n            "order_by": [\n              {\n                "column": "reading_version",\n                "direction": "desc"\n              },\n              {\n                "column": "arrival_timestamp",\n                "direction": "desc"\n              }\n            ]\n          },\n          {\n            "kind": "derive",\n            "name": "resolved_with_hour",\n            "input": "resolved_readings",\n            "columns": [\n              {\n                "name": "event_hour",\n                "expression": "timestampadd(SECOND, -second(event_timestamp), timestampadd(MINUTE, -minute(event_timestamp), event_timestamp))"\n              }\n            ]\n          },\n          {\n            "kind": "aggregate",\n            "name": "hourly_health",\n            "input": "resolved_with_hour",\n            "group_by": [\n              "site_id",\n              "machine_id",\n              "event_hour"\n            ],\n            "measures": [\n              {\n                "name": "total_reading_count",\n                "expression": "count(*)"\n              },\n              {\n                "name": "valid_temperature_count",\n                "expression": "count(temperature_c)"\n              },\n              {\n                "name": "missing_temperature_count",\n                "expression": "sum(case when temperature_c is null then 1 else 0 end)"\n              },\n              {\n                "name": "avg_temperature_c",\n                "expression": "avg(temperature_c)"\n              },\n              {\n                "name": "valid_vibration_count",\n                "expression": "count(vibration_mm_s)"\n              },\n              {\n                "name": "missing_vibration_count",\n                "expression": "sum(case when vibration_mm_s is null then 1 else 0 end)"\n              },\n              {\n                "name": "avg_vibration_mm_s",\n                "expression": "avg(vibration_mm_s)"\n              }\n            ]\n          },\n          {\n            "kind": "filter",\n            "name": "reference_eligible",\n            "input": "readings_raw",\n            "predicate": "arrival_timestamp <= timestampadd(HOUR, 2, event_timestamp)"\n          },\n          {\n            "kind": "deduplicate",\n            "name": "reference_resolved",\n            "input": "reference_eligible",\n            "keys": [\n              "reading_id"\n            ],\n            "order_by": [\n              {\n                "column": "reading_version",\n                "direction": "desc"\n              },\n              {\n                "column": "arrival_timestamp",\n                "direction": "desc"\n              }\n            ]\n          },\n          {\n            "kind": "derive",\n            "name": "reference_with_hour",\n            "input": "reference_resolved",\n            "columns": [\n              {\n                "name": "event_hour",\n                "expression": "timestampadd(SECOND, -second(event_timestamp), timestampadd(MINUTE, -minute(event_timestamp), event_timestamp))"\n              }\n            ]\n          },\n          {\n            "kind": "aggregate",\n            "name": "hourly_reference",\n            "input": "reference_with_hour",\n            "group_by": [\n              "site_id",\n              "machine_id",\n              "event_hour"\n            ],\n            "measures": [\n              {\n                "name": "ref_total_reading_count",\n                "expression": "count(*)"\n              },\n              {\n                "name": "ref_valid_temperature_count",\n                "expression": "count(temperature_c)"\n              },\n              {\n                "name": "ref_missing_temperature_count",\n                "expression": "sum(case when temperature_c is null then 1 else 0 end)"\n              },\n              {\n                "name": "ref_avg_temperature_c",\n                "expression": "avg(temperature_c)"\n              },\n              {\n                "name": "ref_valid_vibration_count",\n                "expression": "count(vibration_mm_s)"\n              },\n              {\n                "name": "ref_missing_vibration_count",\n                "expression": "sum(case when vibration_mm_s is null then 1 else 0 end)"\n              },\n              {\n                "name": "ref_avg_vibration_mm_s",\n                "expression": "avg(vibration_mm_s)"\n              }\n            ]\n          },\n          {\n            "kind": "join",\n            "name": "hourly_comparison",\n            "left": "hourly_health",\n            "right": "hourly_reference",\n            "keys": [\n              "site_id",\n              "machine_id",\n              "event_hour"\n            ],\n            "how": "left",\n            "cardinality": "many_to_one"\n          },\n          {\n            "kind": "filter",\n            "name": "too_late_records",\n            "input": "readings_raw",\n            "predicate": "arrival_timestamp > timestampadd(HOUR, 2, event_timestamp)"\n          },\n          {\n            "kind": "derive",\n            "name": "late_quarantine",\n            "input": "too_late_records",\n            "columns": [\n              {\n                "name": "quarantine_reason",\n                "expression": "\'arrival_after_two_hour_cutoff\'"\n              }\n            ]\n          }\n        ]\n      }\n    },\n    {\n      "key": "population",\n      "value": "Highest eligible version, latest arrival tie-break; arrivals after two hours quarantined.",\n      "status": "confirmed",\n      "source": "User answers",\n      "impact": "Eligibility.",\n      "severity": "requirements",\n      "bindings": {}\n    },\n    {\n      "key": "time",\n      "value": "Fixed UTC window 2026-01-01 through 2026-01-08; two-hour allowed lateness.",\n      "status": "delegated",\n      "source": "User unavailable; reproducibility default",\n      "impact": "Time boundaries.",\n      "severity": "requirements",\n      "bindings": {}\n    },\n    {\n      "key": "simulation",\n      "value": "10,000 base readings, 3 sites, 12 machines, one 45% volume machine, 5% missing temperature, 3% repeats, 2% corrections, 10% delayed and 1% too late.",\n      "status": "confirmed",\n      "source": "User prompt plus delegated numeric defaults",\n      "impact": "Synthetic evidence.",\n      "severity": "requirements",\n      "bindings": {\n        "tables": [\n          {\n            "name": "readings_raw",\n            "grain": "One physical reading record; repeated rows and corrected versions share reading_id",\n            "rows": 10000,\n            "primary_key": "reading_id",\n            "columns": [\n              {\n                "name": "reading_id",\n                "kind": "id"\n              },\n              {\n                "name": "machine_id",\n                "kind": "category",\n                "categories": [\n                  "machine_01",\n                  "machine_02",\n                  "machine_03",\n                  "machine_04",\n                  "machine_05",\n                  "machine_06",\n                  "machine_07",\n                  "machine_08",\n                  "machine_09",\n                  "machine_10",\n                  "machine_11",\n                  "machine_12"\n                ],\n                "weights": [\n                  0.45,\n                  0.05,\n                  0.05,\n                  0.05,\n                  0.05,\n                  0.05,\n                  0.05,\n                  0.05,\n                  0.05,\n                  0.05,\n                  0.05,\n                  0.05\n                ]\n              },\n              {\n                "name": "site_id",\n                "kind": "expression",\n                "expression": "case when machine_id in (\'machine_01\',\'machine_02\',\'machine_03\',\'machine_04\') then \'site_01\' when machine_id in (\'machine_05\',\'machine_06\',\'machine_07\',\'machine_08\') then \'site_02\' else \'site_03\' end"\n              },\n              {\n                "name": "event_timestamp",\n                "kind": "timestamp",\n                "start": "2026-01-01T00:00:00Z",\n                "end": "2026-01-08T00:00:00Z"\n              },\n              {\n                "name": "reading_version",\n                "kind": "integer",\n                "minimum": 1,\n                "maximum": 1\n              },\n              {\n                "name": "temperature_c",\n                "kind": "decimal",\n                "minimum": 40,\n                "maximum": 100,\n                "null_rate": 0.05\n              },\n              {\n                "name": "vibration_mm_s",\n                "kind": "decimal",\n                "minimum": 0,\n                "maximum": 20\n              },\n              {\n                "name": "arrival_class",\n                "kind": "category",\n                "categories": [\n                  "on_time",\n                  "delayed",\n                  "too_late"\n                ],\n                "weights": [\n                  0.89,\n                  0.1,\n                  0.01\n                ]\n              },\n              {\n                "name": "arrival_delay_minutes",\n                "kind": "expression",\n                "expression": "case when arrival_class = \'delayed\' then 90 when arrival_class = \'too_late\' then 180 else 0 end"\n              },\n              {\n                "name": "arrival_timestamp",\n                "kind": "expression",\n                "expression": "timestampadd(MINUTE, arrival_delay_minutes, event_timestamp)"\n              },\n              {\n                "name": "microbatch_id",\n                "kind": "expression",\n                "expression": "floor(timestampdiff(SECOND, to_timestamp(\'2026-01-01T00:00:00Z\'), arrival_timestamp) / 3600)"\n              }\n            ],\n            "duplicate_rate": 0.03,\n            "changed_duplicate_rate": 0.02,\n            "version_column": "reading_version",\n            "duplicate_changes": {\n              "temperature_c": "temperature_c + 1.5",\n              "vibration_mm_s": "vibration_mm_s + 0.25",\n              "arrival_delay_minutes": "arrival_delay_minutes + 30",\n              "arrival_timestamp": "timestampadd(MINUTE, 30, arrival_timestamp)",\n              "microbatch_id": "microbatch_id + 1"\n            }\n          }\n        ]\n      }\n    },\n    {\n      "key": "quality",\n      "value": "Deterministic version resolution and reason-coded late quarantine.",\n      "status": "confirmed",\n      "source": "User answers",\n      "impact": "Clean contract.",\n      "severity": "requirements",\n      "bindings": {}\n    },\n    {\n      "key": "acceptance",\n      "value": "Exact reference equality, identity uniqueness, population reconciliation, anomaly bounds.",\n      "status": "confirmed",\n      "source": "User answer",\n      "impact": "Executable success.",\n      "severity": "requirements",\n      "bindings": {\n        "checks": [\n          {\n            "name": "raw_physical_row_count",\n            "kind": "row_count",\n            "table": "readings_raw",\n            "expected": 10500\n          },\n          {\n            "name": "raw_temperature_null_rate",\n            "kind": "null_rate",\n            "table": "readings_raw",\n            "columns": [\n              "temperature_c"\n            ],\n            "expected": 0.05,\n            "tolerance": 0.02\n          },\n          {\n            "name": "resolved_logical_identity",\n            "kind": "unique",\n            "table": "resolved_readings",\n            "columns": [\n              "reading_id"\n            ]\n          },\n          {\n            "name": "resolved_population_bounds",\n            "kind": "row_count",\n            "table": "resolved_readings",\n            "minimum": 9500,\n            "maximum": 10000\n          },\n          {\n            "name": "hourly_metric_reconciliation",\n            "kind": "predicate",\n            "table": "hourly_health",\n            "predicate": "total_reading_count = valid_temperature_count + missing_temperature_count and total_reading_count = valid_vibration_count + missing_vibration_count",\n            "maximum_failures": 0\n          },\n          {\n            "name": "hourly_reference_exact_match",\n            "kind": "predicate",\n            "table": "hourly_comparison",\n            "predicate": "total_reading_count = ref_total_reading_count and valid_temperature_count = ref_valid_temperature_count and missing_temperature_count = ref_missing_temperature_count and coalesce(avg_temperature_c, -9999) = coalesce(ref_avg_temperature_c, -9999) and valid_vibration_count = ref_valid_vibration_count and missing_vibration_count = ref_missing_vibration_count and coalesce(avg_vibration_mm_s, -9999) = coalesce(ref_avg_vibration_mm_s, -9999)",\n            "maximum_failures": 0\n          },\n          {\n            "name": "late_quarantine_present",\n            "kind": "row_count",\n            "table": "late_quarantine",\n            "minimum": 1,\n            "maximum": 500\n          },\n          {\n            "name": "late_quarantine_reason",\n            "kind": "predicate",\n            "table": "late_quarantine",\n            "predicate": "quarantine_reason = \'arrival_after_two_hour_cutoff\' and arrival_timestamp > timestampadd(HOUR, 2, event_timestamp)",\n            "maximum_failures": 0\n          }\n        ]\n      }\n    },\n    {\n      "key": "pipeline",\n      "value": "bounded batch",\n      "status": "confirmed",\n      "source": "User answer and supported-scope boundary",\n      "impact": "Execution semantics.",\n      "severity": "requirements",\n      "bindings": {}\n    },\n    {\n      "key": "persistence",\n      "value": "Disabled by default and runtime opt-in only.",\n      "status": "confirmed",\n      "source": "User prompt",\n      "impact": "Write safety.",\n      "severity": "runtime",\n      "bindings": {}\n    }\n  ],\n  "provisional": false,\n  "provisional_reason": "",\n  "verify_reproducibility": true,\n  "scenario": "pipeline",\n  "tables": [\n    {\n      "name": "readings_raw",\n      "grain": "One physical reading record; repeated rows and corrected versions share reading_id",\n      "rows": 10000,\n      "primary_key": "reading_id",\n      "columns": [\n        {\n          "name": "reading_id",\n          "kind": "id",\n          "minimum": 0,\n          "maximum": 100,\n          "categories": [],\n          "weights": [],\n          "true_rate": 0.5,\n          "null_rate": 0.0,\n          "start": null,\n          "end": null,\n          "reference_table": null,\n          "expression": null\n        },\n        {\n          "name": "machine_id",\n          "kind": "category",\n          "minimum": 0,\n          "maximum": 100,\n          "categories": [\n            "machine_01",\n            "machine_02",\n            "machine_03",\n            "machine_04",\n            "machine_05",\n            "machine_06",\n            "machine_07",\n            "machine_08",\n            "machine_09",\n            "machine_10",\n            "machine_11",\n            "machine_12"\n          ],\n          "weights": [\n            0.45,\n            0.05,\n            0.05,\n            0.05,\n            0.05,\n            0.05,\n            0.05,\n            0.05,\n            0.05,\n            0.05,\n            0.05,\n            0.05\n          ],\n          "true_rate": 0.5,\n          "null_rate": 0.0,\n          "start": null,\n          "end": null,\n          "reference_table": null,\n          "expression": null\n        },\n        {\n          "name": "site_id",\n          "kind": "expression",\n          "minimum": 0,\n          "maximum": 100,\n          "categories": [],\n          "weights": [],\n          "true_rate": 0.5,\n          "null_rate": 0.0,\n          "start": null,\n          "end": null,\n          "reference_table": null,\n          "expression": "case when machine_id in (\'machine_01\',\'machine_02\',\'machine_03\',\'machine_04\') then \'site_01\' when machine_id in (\'machine_05\',\'machine_06\',\'machine_07\',\'machine_08\') then \'site_02\' else \'site_03\' end"\n        },\n        {\n          "name": "event_timestamp",\n          "kind": "timestamp",\n          "minimum": 0,\n          "maximum": 100,\n          "categories": [],\n          "weights": [],\n          "true_rate": 0.5,\n          "null_rate": 0.0,\n          "start": "2026-01-01T00:00:00Z",\n          "end": "2026-01-08T00:00:00Z",\n          "reference_table": null,\n          "expression": null\n        },\n        {\n          "name": "reading_version",\n          "kind": "integer",\n          "minimum": 1,\n          "maximum": 1,\n          "categories": [],\n          "weights": [],\n          "true_rate": 0.5,\n          "null_rate": 0.0,\n          "start": null,\n          "end": null,\n          "reference_table": null,\n          "expression": null\n        },\n        {\n          "name": "temperature_c",\n          "kind": "decimal",\n          "minimum": 40,\n          "maximum": 100,\n          "categories": [],\n          "weights": [],\n          "true_rate": 0.5,\n          "null_rate": 0.05,\n          "start": null,\n          "end": null,\n          "reference_table": null,\n          "expression": null\n        },\n        {\n          "name": "vibration_mm_s",\n          "kind": "decimal",\n          "minimum": 0,\n          "maximum": 20,\n          "categories": [],\n          "weights": [],\n          "true_rate": 0.5,\n          "null_rate": 0.0,\n          "start": null,\n          "end": null,\n          "reference_table": null,\n          "expression": null\n        },\n        {\n          "name": "arrival_class",\n          "kind": "category",\n          "minimum": 0,\n          "maximum": 100,\n          "categories": [\n            "on_time",\n            "delayed",\n            "too_late"\n          ],\n          "weights": [\n            0.89,\n            0.1,\n            0.01\n          ],\n          "true_rate": 0.5,\n          "null_rate": 0.0,\n          "start": null,\n          "end": null,\n          "reference_table": null,\n          "expression": null\n        },\n        {\n          "name": "arrival_delay_minutes",\n          "kind": "expression",\n          "minimum": 0,\n          "maximum": 100,\n          "categories": [],\n          "weights": [],\n          "true_rate": 0.5,\n          "null_rate": 0.0,\n          "start": null,\n          "end": null,\n          "reference_table": null,\n          "expression": "case when arrival_class = \'delayed\' then 90 when arrival_class = \'too_late\' then 180 else 0 end"\n        },\n        {\n          "name": "arrival_timestamp",\n          "kind": "expression",\n          "minimum": 0,\n          "maximum": 100,\n          "categories": [],\n          "weights": [],\n          "true_rate": 0.5,\n          "null_rate": 0.0,\n          "start": null,\n          "end": null,\n          "reference_table": null,\n          "expression": "timestampadd(MINUTE, arrival_delay_minutes, event_timestamp)"\n        },\n        {\n          "name": "microbatch_id",\n          "kind": "expression",\n          "minimum": 0,\n          "maximum": 100,\n          "categories": [],\n          "weights": [],\n          "true_rate": 0.5,\n          "null_rate": 0.0,\n          "start": null,\n          "end": null,\n          "reference_table": null,\n          "expression": "floor(timestampdiff(SECOND, to_timestamp(\'2026-01-01T00:00:00Z\'), arrival_timestamp) / 3600)"\n        }\n      ],\n      "duplicate_rate": 0.03,\n      "changed_duplicate_rate": 0.02,\n      "version_column": "reading_version",\n      "duplicate_changes": {\n        "temperature_c": "temperature_c + 1.5",\n        "vibration_mm_s": "vibration_mm_s + 0.25",\n        "arrival_delay_minutes": "arrival_delay_minutes + 30",\n        "arrival_timestamp": "timestampadd(MINUTE, 30, arrival_timestamp)",\n        "microbatch_id": "microbatch_id + 1"\n      }\n    }\n  ],\n  "operations": [\n    {\n      "kind": "filter",\n      "name": "eligible_readings",\n      "input": "readings_raw",\n      "predicate": "arrival_delay_minutes <= 120"\n    },\n    {\n      "kind": "deduplicate",\n      "name": "resolved_readings",\n      "input": "eligible_readings",\n      "keys": [\n        "reading_id"\n      ],\n      "order_by": [\n        {\n          "column": "reading_version",\n          "direction": "desc"\n        },\n        {\n          "column": "arrival_timestamp",\n          "direction": "desc"\n        }\n      ]\n    },\n    {\n      "kind": "derive",\n      "name": "resolved_with_hour",\n      "input": "resolved_readings",\n      "columns": [\n        {\n          "name": "event_hour",\n          "expression": "timestampadd(SECOND, -second(event_timestamp), timestampadd(MINUTE, -minute(event_timestamp), event_timestamp))"\n        }\n      ]\n    },\n    {\n      "kind": "aggregate",\n      "name": "hourly_health",\n      "input": "resolved_with_hour",\n      "group_by": [\n        "site_id",\n        "machine_id",\n        "event_hour"\n      ],\n      "measures": [\n        {\n          "name": "total_reading_count",\n          "expression": "count(*)"\n        },\n        {\n          "name": "valid_temperature_count",\n          "expression": "count(temperature_c)"\n        },\n        {\n          "name": "missing_temperature_count",\n          "expression": "sum(case when temperature_c is null then 1 else 0 end)"\n        },\n        {\n          "name": "avg_temperature_c",\n          "expression": "avg(temperature_c)"\n        },\n        {\n          "name": "valid_vibration_count",\n          "expression": "count(vibration_mm_s)"\n        },\n        {\n          "name": "missing_vibration_count",\n          "expression": "sum(case when vibration_mm_s is null then 1 else 0 end)"\n        },\n        {\n          "name": "avg_vibration_mm_s",\n          "expression": "avg(vibration_mm_s)"\n        }\n      ]\n    },\n    {\n      "kind": "filter",\n      "name": "reference_eligible",\n      "input": "readings_raw",\n      "predicate": "arrival_timestamp <= timestampadd(HOUR, 2, event_timestamp)"\n    },\n    {\n      "kind": "deduplicate",\n      "name": "reference_resolved",\n      "input": "reference_eligible",\n      "keys": [\n        "reading_id"\n      ],\n      "order_by": [\n        {\n          "column": "reading_version",\n          "direction": "desc"\n        },\n        {\n          "column": "arrival_timestamp",\n          "direction": "desc"\n        }\n      ]\n    },\n    {\n      "kind": "derive",\n      "name": "reference_with_hour",\n      "input": "reference_resolved",\n      "columns": [\n        {\n          "name": "event_hour",\n          "expression": "timestampadd(SECOND, -second(event_timestamp), timestampadd(MINUTE, -minute(event_timestamp), event_timestamp))"\n        }\n      ]\n    },\n    {\n      "kind": "aggregate",\n      "name": "hourly_reference",\n      "input": "reference_with_hour",\n      "group_by": [\n        "site_id",\n        "machine_id",\n        "event_hour"\n      ],\n      "measures": [\n        {\n          "name": "ref_total_reading_count",\n          "expression": "count(*)"\n        },\n        {\n          "name": "ref_valid_temperature_count",\n          "expression": "count(temperature_c)"\n        },\n        {\n          "name": "ref_missing_temperature_count",\n          "expression": "sum(case when temperature_c is null then 1 else 0 end)"\n        },\n        {\n          "name": "ref_avg_temperature_c",\n          "expression": "avg(temperature_c)"\n        },\n        {\n          "name": "ref_valid_vibration_count",\n          "expression": "count(vibration_mm_s)"\n        },\n        {\n          "name": "ref_missing_vibration_count",\n          "expression": "sum(case when vibration_mm_s is null then 1 else 0 end)"\n        },\n        {\n          "name": "ref_avg_vibration_mm_s",\n          "expression": "avg(vibration_mm_s)"\n        }\n      ]\n    },\n    {\n      "kind": "join",\n      "name": "hourly_comparison",\n      "left": "hourly_health",\n      "right": "hourly_reference",\n      "keys": [\n        "site_id",\n        "machine_id",\n        "event_hour"\n      ],\n      "how": "left",\n      "cardinality": "many_to_one"\n    },\n    {\n      "kind": "filter",\n      "name": "too_late_records",\n      "input": "readings_raw",\n      "predicate": "arrival_timestamp > timestampadd(HOUR, 2, event_timestamp)"\n    },\n    {\n      "kind": "derive",\n      "name": "late_quarantine",\n      "input": "too_late_records",\n      "columns": [\n        {\n          "name": "quarantine_reason",\n          "expression": "\'arrival_after_two_hour_cutoff\'"\n        }\n      ]\n    }\n  ],\n  "outputs": [\n    "readings_raw",\n    "resolved_readings",\n    "hourly_health",\n    "hourly_comparison",\n    "late_quarantine"\n  ],\n  "checks": [\n    {\n      "name": "raw_physical_row_count",\n      "kind": "row_count",\n      "table": "readings_raw",\n      "columns": [],\n      "expected": 10500,\n      "minimum": null,\n      "maximum": null,\n      "tolerance": 0.0,\n      "reference_table": null,\n      "reference_columns": [],\n      "predicate": null,\n      "maximum_failures": 0\n    },\n    {\n      "name": "raw_temperature_null_rate",\n      "kind": "null_rate",\n      "table": "readings_raw",\n      "columns": [\n        "temperature_c"\n      ],\n      "expected": 0.05,\n      "minimum": null,\n      "maximum": null,\n      "tolerance": 0.02,\n      "reference_table": null,\n      "reference_columns": [],\n      "predicate": null,\n      "maximum_failures": 0\n    },\n    {\n      "name": "resolved_logical_identity",\n      "kind": "unique",\n      "table": "resolved_readings",\n      "columns": [\n        "reading_id"\n      ],\n      "expected": null,\n      "minimum": null,\n      "maximum": null,\n      "tolerance": 0.0,\n      "reference_table": null,\n      "reference_columns": [],\n      "predicate": null,\n      "maximum_failures": 0\n    },\n    {\n      "name": "resolved_population_bounds",\n      "kind": "row_count",\n      "table": "resolved_readings",\n      "columns": [],\n      "expected": null,\n      "minimum": 9500,\n      "maximum": 10000,\n      "tolerance": 0.0,\n      "reference_table": null,\n      "reference_columns": [],\n      "predicate": null,\n      "maximum_failures": 0\n    },\n    {\n      "name": "hourly_metric_reconciliation",\n      "kind": "predicate",\n      "table": "hourly_health",\n      "columns": [],\n      "expected": null,\n      "minimum": null,\n      "maximum": null,\n      "tolerance": 0.0,\n      "reference_table": null,\n      "reference_columns": [],\n      "predicate": "total_reading_count = valid_temperature_count + missing_temperature_count and total_reading_count = valid_vibration_count + missing_vibration_count",\n      "maximum_failures": 0\n    },\n    {\n      "name": "hourly_reference_exact_match",\n      "kind": "predicate",\n      "table": "hourly_comparison",\n      "columns": [],\n      "expected": null,\n      "minimum": null,\n      "maximum": null,\n      "tolerance": 0.0,\n      "reference_table": null,\n      "reference_columns": [],\n      "predicate": "total_reading_count = ref_total_reading_count and valid_temperature_count = ref_valid_temperature_count and missing_temperature_count = ref_missing_temperature_count and coalesce(avg_temperature_c, -9999) = coalesce(ref_avg_temperature_c, -9999) and valid_vibration_count = ref_valid_vibration_count and missing_vibration_count = ref_missing_vibration_count and coalesce(avg_vibration_mm_s, -9999) = coalesce(ref_avg_vibration_mm_s, -9999)",\n      "maximum_failures": 0\n    },\n    {\n      "name": "late_quarantine_present",\n      "kind": "row_count",\n      "table": "late_quarantine",\n      "columns": [],\n      "expected": null,\n      "minimum": 1,\n      "maximum": 500,\n      "tolerance": 0.0,\n      "reference_table": null,\n      "reference_columns": [],\n      "predicate": null,\n      "maximum_failures": 0\n    },\n    {\n      "name": "late_quarantine_reason",\n      "kind": "predicate",\n      "table": "late_quarantine",\n      "columns": [],\n      "expected": null,\n      "minimum": null,\n      "maximum": null,\n      "tolerance": 0.0,\n      "reference_table": null,\n      "reference_columns": [],\n      "predicate": "quarantine_reason = \'arrival_after_two_hour_cutoff\' and arrival_timestamp > timestampadd(HOUR, 2, event_timestamp)",\n      "maximum_failures": 0\n    }\n  ],\n  "scale_acknowledged": false\n}')
SPEC_FINGERPRINT = '0268972c86dc651f85506684e2e34af0bbf4a9826ccf5d76a08d62acd1550f6c'


In [ ]:
for key in ['target_catalog', 'target_schema', 'table_prefix']:
    dbutils.widgets.text(key, '')
target_catalog = dbutils.widgets.get('target_catalog')
target_schema = dbutils.widgets.get('target_schema')
table_prefix = dbutils.widgets.get('table_prefix')
for value in [target_catalog, target_schema, table_prefix]:
    if not re.fullmatch(r'[A-Za-z][A-Za-z0-9_]{0,62}', value):
        raise ValueError('Set catalog, schema and prefix to the values used by the generator')


In [ ]:
targets = {key: f'{target_catalog}.{target_schema}.{table_prefix}_{key}' for key in cfg['outputs']}
if any(len(target.rsplit('.', 1)[1]) > 63 for target in targets.values()):
    raise ValueError('Combined table name must be <=63 characters')
missing = [target for target in targets.values() if not spark.catalog.tableExists(target)]
if missing:
    raise ValueError('Missing saved outputs: ' + ', '.join(missing))
tables = {key: spark.read.format('delta').table(target) for key, target in targets.items()}


## Read execution
Delta reads start new lineage. Validation actions can scan repeatedly. Inspect storage pruning and the executed profile rather than assuming timing.

## Inspect the execution plan
explain(formatted) requests planning evidence, not a row scan. Look for Exchange (redistribution), BroadcastHashJoin/BroadcastExchange, and aggregate operators. Names may differ by runtime/engine. Adaptive execution can change the final plan: inspect the executed query profile after an action for task skew, spills, shuffle volume and actual timings. This cell does not measure partition counts or prove that a suggested optimization helps. Serverless uses DataFrame APIs; no RDD introspection is used.

References: [Spark execution](https://spark.apache.org/docs/latest/rdd-programming-guide.html) · [Databricks serverless limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations).

In [ ]:
for name in cfg['outputs']:
    print('Plan:', name)
    tables[name].explain('formatted')


In [ ]:
def validate_pipeline(tables, cfg):
    from functools import reduce

    missing = set(cfg["outputs"]) - set(tables)
    if missing:
        raise ValueError("Missing declared outputs: " + ", ".join(sorted(missing)))
    from pyspark.sql import functions as F

    counts = {name: tables[name].count() for name in cfg["outputs"]}
    checks = []
    for check in cfg["checks"]:
        frame = tables[check["table"]]
        kind = check["kind"]
        entry = {"name": check["name"], "kind": kind, "table": check["table"]}
        if kind == "row_count":
            actual = counts[check["table"]]
            passed = check["expected"] is None or actual == check["expected"]
            passed = passed and (check["minimum"] is None or actual >= check["minimum"])
            passed = passed and (check["maximum"] is None or actual <= check["maximum"])
        elif kind == "unique":
            columns = check["columns"]
            duplicates = frame.groupBy(*columns).count().filter(F.col("count") > 1).count()
            nulls = frame.filter(reduce(lambda a, b: a | b, [F.col(key).isNull() for key in columns])).count()
            actual = {"duplicate_groups": duplicates, "null_key_rows": nulls}
            passed = duplicates == 0 and nulls == 0
        elif kind == "null_rate":
            nulls = frame.filter(F.col(check["columns"][0]).isNull()).count()
            actual = nulls / counts[check["table"]] if counts[check["table"]] else 0.0
            passed = abs(actual - check["expected"]) <= check["tolerance"]
        elif kind == "foreign_key":
            columns = check["columns"]
            reference = tables[check["reference_table"]]
            valid = reduce(lambda a, b: a & b, [F.col(key).isNotNull() for key in columns])
            keys = ["join_key_" + str(index) for index in range(len(columns))]
            left = frame.filter(valid).select(*[F.col(key).alias(alias) for key, alias in zip(columns, keys)])
            right = reference.select(
                *[F.col(key).alias(alias) for key, alias in zip(check["reference_columns"], keys)]
            ).distinct()
            actual = left.join(right, keys, "left_anti").count()
            passed = actual <= check["maximum_failures"]
        elif kind == "predicate":
            actual = frame.filter(
                ~F.coalesce(F.expr(check["predicate"]).cast("boolean"), F.lit(False))
            ).count()
            passed = actual <= check["maximum_failures"]
        else:
            raise ValueError("Unsupported acceptance check")
        entry.update(passed=bool(passed), actual=actual)
        checks.append(entry)
    return {"passed": all(check["passed"] for check in checks), "counts": counts, "checks": checks}

summary = validate_pipeline(tables, cfg)
summary['spec_fingerprint'] = SPEC_FINGERPRINT
summary['generator_version'] = '0.1.0'
print(json.dumps(summary, indent=2))
if not summary['passed']:
    raise ValueError('Acceptance checks failed; inspect summary')


In [ ]:
for name, frame in tables.items():
    print(name)
    display(frame.limit(20))


In [ ]:
summary['saved_tables'] = targets
if 'dbutils' in globals():
    dbutils.notebook.exit(json.dumps(summary))
